# AI Sommelier RAG: 와인 리뷰 인덱싱

이 노트북은 RAG의 사전 준비 단계인 **인덱싱**을 다룬다. Wine Magazine 리뷰를 의미로 검색할 수 있게 변환해 Pinecone에 저장한다.

### 인덱싱 흐름

`CSV 행 → Document → embedding vector → Pinecone index`

### 핵심 구성 요소

- `Document`: 검색 본문 `page_content`와 출처 정보 `metadata`를 함께 담는다.
- `Embedding`: 문서 본문의 의미를 숫자 vector로 변환한다.
- `Pinecone index`: vector와 원문·metadata를 저장하고 검색한다.
- `namespace`: 하나의 index 안에서 레코드를 논리적으로 구분한다.

### 인덱싱과 검색의 차이

- 인덱싱: 문서를 미리 변환해 저장하는 작업이다.
- Querying: 사용자 질문이 들어오면 관련 문서를 찾는 작업이다.

다음 노트북은 여기서 만든 `winemag-review-data` index를 조회한다. 검색 결과인 `list[Document]`는 와인 추천 Prompt의 context로 사용한다.


## Pinecone이란

Pinecone은 embedding vector를 저장하고 의미가 가까운 데이터를 검색하는 **관리형 Vector DB**이다. 사용자가 별도 데이터베이스 서버를 설치하지 않아도 Pinecone이 원격 저장소의 운영, 확장과 API 접속을 관리한다.

### Chroma와의 차이

- `Chroma`: 개인 PC에서 빠르게 시작하기 좋은 로컬 Vector Store이다. 폴더에 저장하면 같은 PC에서 다시 열 수 있다.
- `Pinecone`: 인터넷을 통해 접속하는 클라우드 Vector DB이다. 여러 실행 환경이 같은 index를 사용할 수 있지만 계정과 API Key가 필요하다.

### 이 실습에서 Pinecone이 하는 일

`Document → OpenAI embedding vector → Pinecone index → 유사도 검색 → list[Document]`

1. `OpenAIEmbeddings`가 `Document.page_content`를 숫자 vector로 변환한다.
2. Pinecone index가 vector와 원문·metadata를 함께 저장한다.
3. 질문이 들어오면 같은 Embedding Model로 query vector를 만든다.
4. Pinecone이 가까운 vector를 찾고 연결된 Document를 반환한다.

Pinecone은 LLM 답변을 생성하지 않는다. 검색된 `list[Document]`를 다음 RAG 단계에서 Prompt의 context로 전달한다. [Pinecone Quickstart](https://docs.pinecone.io/guides/get-started/quickstart)에서 전체 저장·검색 흐름을 확인할 수 있다.


## 1. Pinecone 가입과 프로젝트 선택

먼저 [Pinecone Console](https://app.pinecone.io/)에 가입하고 로그인한다. Pinecone의 Index와 API Key는 프로젝트에 속하므로 수업에서 사용할 프로젝트를 먼저 정해야 한다.

### 준비 순서

1. Pinecone Console에 가입하고 로그인한다.
2. 수업에 사용할 프로젝트를 선택하거나 새 프로젝트를 만든다.
3. 이후 Index 생성과 API Key 발급을 모두 같은 프로젝트에서 진행한다.

프로젝트를 바꾸면 Index 목록과 API Key도 달라질 수 있다. 뒤에서 Index를 찾지 못하는 오류가 발생하면 현재 선택한 프로젝트부터 확인한다.


## 2. 수업용 Pinecone Index 생성

이 실습은 Pinecone이 텍스트를 직접 임베딩하는 방식이 아니라, OpenAI가 만든 vector를 Pinecone에 저장하는 **외부 임베딩 방식**을 사용한다. 따라서 코드를 실행하기 전에 Pinecone Console에서 두 개의 dense index를 준비한다.


### 공통 설정

- 방식: `Bring your own vectors` 또는 외부 embedding vector를 저장하는 설정
- Vector type: `Dense`
- Dimension: `1536`
- Metric: `cosine`
- 배포 방식: 계정에서 선택 가능한 Serverless cloud와 region

`text-embedding-3-small`을 별도 `dimensions` 인자 없이 호출하면 1536차원 vector를 반환하므로 Index dimension도 1536이어야 한다. Dimension이 다르면 문서를 저장하거나 검색할 때 오류가 발생한다.

### Console에서 만드는 순서

1. Pinecone Console의 `Indexes`에서 `Create index`를 선택한다.
2. Index 이름에 `pinecone-first`를 입력한다.
3. 위 공통 설정으로 생성하고 상태가 `Ready`가 될 때까지 기다린다.
4. 같은 설정으로 `winemag-review-data`도 생성한다.
5. 두 Index가 현재 선택한 프로젝트에 생성되었는지 확인한다.

코드의 `DEMO_INDEX_NAME`과 `WINE_INDEX_NAME`은 이 이름을 그대로 사용한다. 다른 이름으로 만들었다면 코드의 상수도 같은 이름으로 바꿔야 한다. Index 생성 방식과 옵션은 [Pinecone Index 공식 가이드](https://docs.pinecone.io/guides/index-data/create-an-index)에서 확인할 수 있다.


## 3. Pinecone API Key 발급과 `.env` 등록

API Key는 Python 코드가 Pinecone 프로젝트에 접속할 때 사용하는 인증 정보이다. 앞에서 Index를 만든 프로젝트에서 API Key를 발급해야 코드가 해당 Index를 찾을 수 있다.

### API Key 발급 순서

1. Pinecone Console에서 앞서 Index를 만든 프로젝트를 선택한다.
2. `API keys` 메뉴를 연다.
3. `Create API key`를 선택하고 `skn-llm-lecture`처럼 용도를 알 수 있는 이름을 입력한다.
4. 생성된 API Key를 복사한다.
5. PyCharm 프로젝트의 `.env` 파일에 아래와 같이 등록한다.

```text
OPENAI_API_KEY=your_openai_api_key
OPENAI_EMBEDDING_MODEL=text-embedding-3-small
PINECONE_API_KEY=your_pinecone_api_key
```

- `PINECONE_API_KEY`는 앞에서 Index를 만든 프로젝트의 Key를 사용한다.
- `.env`는 Git에 올리지 않는다.
- API Key를 코드, 출력, 화면 캡처에 포함하지 않는다.
- 키가 노출되면 Pinecone Console에서 삭제하고 새로 발급한다.

자세한 발급·삭제 절차는 [Pinecone API Key 공식 가이드](https://docs.pinecone.io/guides/projects/manage-api-keys)에서 확인할 수 있다.


## 패키지 준비

이 실습은 LangChain의 `Document`와 OpenAI embedding을 Pinecone에 연결한다.

- `langchain`: `Document`와 Retriever 같은 공통 인터페이스를 제공한다.
- `langchain-openai`: OpenAI embedding 모델을 연결한다.
- `langchain-pinecone`: LangChain `Document`와 Pinecone을 연결한다.
- `pinecone`: Pinecone index에 접속하는 공식 SDK이다.
- `langchain-community`: CSV 행을 `Document`로 읽는 `CSVLoader`를 제공한다.
- `gdown`: 수업용 공개 CSV를 Google Drive에서 내려받는다.
- `python-dotenv`: `.env`의 API 설정을 환경 변수로 불러온다.


In [2]:
# %pip install -U langchain langchain-openai langchain-pinecone langchain-community pinecone gdown python-dotenv


## `.env`의 API 설정 불러오기

앞에서 `.env`에 등록한 값을 `load_dotenv()`로 현재 Python 프로세스의 환경 변수에 불러온다. API Key의 실제 값은 출력하지 않는다.

### 이어지는 코드에서 사용하는 설정

- `OPENAI_API_KEY`: 문서와 query를 embedding할 때 사용한다.
- `OPENAI_EMBEDDING_MODEL`: 사용할 embedding 모델 이름이다.
- `PINECONE_API_KEY`: Pinecone index에 접속할 때 사용한다.

### 반드시 일치해야 하는 설정

- 문서 인덱싱과 query 검색은 같은 embedding 모델을 사용한다.
- Pinecone index의 차원은 embedding vector의 차원과 같아야 한다.
- 이 실습은 `text-embedding-3-small`과 1536차원 Index를 기준으로 사용한다.


In [3]:
import os

from dotenv import load_dotenv

load_dotenv()

DEMO_INDEX_NAME = 'pinecone-first'

WINE_INDEX_NAME = 'winemag-review-data'


## 작은 문서로 Pinecone 인덱싱 흐름 확인하기

Pinecone은 embedding vector를 저장하고 유사도 검색을 수행하는 관리형 Vector DB이다.

### Pinecone에서 사용하는 단위

- `index`: 같은 차원과 유사도 기준을 사용하는 vector 검색 공간이다.
- `namespace`: 하나의 index 안에서 레코드를 논리적으로 나누는 구획이다.
- 기본 namespace: `namespace`를 지정하지 않았을 때 사용하는 저장 구획이다.

`PineconeVectorStore`는 LangChain과 Pinecone을 연결한다. `Document.page_content`를 embedding하고 vector·원문·metadata를 함께 저장하거나 검색한다.

이 실습은 기본 namespace를 사용한다. 별도 namespace를 선택하면 저장과 검색에 같은 namespace를 지정해야 한다.


### `Document` 목록 만들기

`Document`는 Pinecone에 저장할 검색 단위 하나를 표현한다.

- `page_content`: embedding과 의미 검색에 사용하는 본문이다.
- `metadata`: 출처 표시와 조건 필터링에 사용하는 부가 정보이다.
- `documents`: 여러 `Document`를 저장한 목록이다.

목록의 원소 하나가 Pinecone의 검색 레코드 하나가 된다. 필터에 사용할 metadata key와 자료형은 문서마다 같은 규칙으로 작성하는 것이 좋다.


In [9]:
from langchain_core.documents import Document

documents = [
    Document(page_content="LangChain은 LLM 기반 애플리케이션을 쉽게 만들 수 있는 프레임워크입니다.", metadata={"source": "https://langchain.com/docs", "author": "alice", "page": 1}),
    Document(page_content="ChromaDB는 오픈소스 벡터 데이터베이스입니다.", metadata={"source": "https://chromadb.org/intro", "license": "MIT", "date": "2024-07-01"}),
    Document(page_content="파이썬으로 AI 서비스를 개발할 수 있습니다.", metadata={"source": "https://pythonai.co.kr", "editor": "kim", "page": 7}),
    Document(page_content="LLM은 자연어 처리를 위한 대형 언어 모델을 의미합니다.", metadata={"source": "https://llmwiki.com/info", "author": "bob", "version": "v1.1"}),
    Document(page_content="RAG는 검색과 생성의 결합 방식을 제공합니다.", metadata={"source": "https://rag-search.io", "reviewer": "lee", "section": "summary"}),
    Document(page_content="벡터 데이터베이스는 임베딩된 데이터를 효율적으로 검색할 수 있습니다.", metadata={"source": "https://vectorbase.net", "author": "jin", "topic": "vector"}),
    Document(page_content="LangChain을 이용하면 다양한 AI 파이프라인을 구축할 수 있습니다.", metadata={"source": "https://langchain.com/blog", "editor": "sarah", "date": "2024-06-30"}),
    Document(page_content="OpenAI의 GPT 모델은 텍스트 생성에 특화되어 있습니다.", metadata={"source": "https://openai.com/gpt", "lang": "ko", "page": 5}),
    Document(page_content="파이썬은 AI 및 데이터 분석 분야에서 널리 사용되는 언어입니다.", metadata={"source": "https://python.org/usecases", "author": "chun", "updated": "2024-05"}),
    Document(page_content="Streamlit은 파이썬으로 대시보드를 쉽게 만들 수 있는 프레임워크입니다.", metadata={"source": "https://streamlit.io/start", "editor": "park", "date": "2024-04-28"}),
    Document(page_content="Dense Retrieval은 임베딩 벡터를 이용한 검색 방식을 의미합니다.", metadata={"source": "https://retrieval.ai/dense", "type": "tech", "page": 3}),
    Document(page_content="Pandas 라이브러리는 데이터 분석에 자주 사용됩니다.", metadata={"source": "https://pandas.pydata.org/about", "maintainer": "koh", "section": "intro"}),
    Document(page_content="메타데이터 필터링은 검색 결과의 품질을 높여줍니다.", metadata={"source": "https://search.com/metadata", "author": "seo", "feature": "filter"}),
    Document(page_content="SelfQueryRetriever는 자연어 쿼리를 임베딩 쿼리로 변환해줍니다.", metadata={"source": "https://selfquery.ai", "editor": "min", "date": "2024-05-12"}),
    Document(page_content="프롬프트 엔지니어링은 LLM의 성능을 극대화하는 방법입니다.", metadata={"source": "https://prompting.dev/guide", "author": "yang", "topic": "prompt"}),
    Document(page_content="HyDE 기법은 하이브리드 검색에 사용됩니다.", metadata={"source": "https://hyde-tech.com", "reviewer": "kang", "version": "2024.1"}),
    Document(page_content="CoT는 복잡한 문제를 단계적으로 해결하는 프롬프트 기법입니다.", metadata={"source": "https://cotprompt.org", "editor": "jung", "date": "2023-12-01"}),
    Document(page_content="문서 임베딩은 텍스트를 고차원 벡터로 변환하는 과정입니다.", metadata={"source": "https://embedding.ai/intro", "section": "embedding", "author": "song"}),
    Document(page_content="CrewAI는 멀티 에이전트 시스템 구현을 돕는 툴입니다.", metadata={"source": "https://crew.ai/docs", "lang": "ko", "page": 9}),
    Document(page_content="Fine-tuning은 사전학습 모델을 특정 도메인에 맞게 재학습시키는 과정입니다.", metadata={"source": "https://finetune.ai/guide", "editor": "jeon", "date": "2024-01-30"}),
]

### 문서를 embedding해 index에 저장하기

`PineconeVectorStore.from_documents()`는 `Document` 목록으로 Vector Store를 만들고 문서를 바로 저장한다.

### 처리 순서

1. `OpenAIEmbeddings`가 각 `page_content`를 embedding vector로 변환한다.
2. Pinecone에 vector와 원문·metadata를 함께 저장한다.
3. 검색에 사용할 `PineconeVectorStore` 객체를 반환한다.

### 실행 시 주의할 점

- OpenAI Embedding API 비용과 Pinecone 쓰기가 발생한다.
- 같은 셀을 다시 실행하면 같은 문서가 새 ID로 중복 저장될 수 있다.
- 실제 적재 개수는 Pinecone index 통계 또는 다음 검색 결과로 확인한다.


In [11]:
from langchain_openai import OpenAIEmbeddings

from langchain_pinecone import PineconeVectorStore

# 1. Document와 사용자 질분(query)를 벡터화 시킬 Embedding 객체 생성
embeddings = OpenAIEmbeddings(
    model=os.environ['OPENAI_EMBEDDING_MODEL']
)

# 2. Document를 embedding하고 Pinecone에 저장하기
vector_store = PineconeVectorStore.from_documents(
    documents=documents, # 저장할 문서 목록
    embedding=embeddings, # 문서, query 임베딩 시킬때 사용할 객체
    index_name=DEMO_INDEX_NAME # pinecone-first 인덱스 선텍
)





### 기존 index를 Retriever로 검색하기

`Retriever`는 자연어 query를 받아 관련 `Document`를 찾는 검색 인터페이스이다. 답변을 생성하지 않는다.

### 검색 흐름

1. query 문자열을 embedding vector로 변환한다.
2. 저장된 문서 vector와 유사도를 비교한다.
3. 상위 `k`개의 `Document`를 반환한다.

- 입력: 자연어 query 문자열이다.
- 출력: `list[Document]`이다.
- 다음 사용처: 각 문서의 `page_content`를 RAG Prompt의 context로 전달한다.

`as_retriever()`는 Vector Store의 검색 기능을 이 공통 인터페이스로 바꾼다.


In [15]:
from pinecone import Pinecone

# 1. Pinecone SDK client를 만들고, API키로 계정 인증
pinecone_client = Pinecone(
    api_key=os.environ['PINECONE_API_KEY']
)

# 2. 'pinecone-fist' 인덱스과 연결된 객체 반환받기
pinecone_index = pinecone_client.Index(DEMO_INDEX_NAME)

# 3. 원격 인덱스와 임베딩 객체를 이용해서 LangChain VectorStore 생성
vector_store = PineconeVectorStore(
    embedding=embeddings, # 문서,query 임베딩 시킬 사용할 객체
    index=pinecone_index # pinecone-first 인덱스 선택
)

# 4. LangChain VectorStore -> Retriever로 변환
# - 자연어 query -> Vector Store 검색 -> list[Document] 반환
retriever = vector_store.as_retriever(
    search_type = 'similarity', # query 벡터와 가까운 문서 벡터 검색
    search_kwargs={'k': 3}  # 제일 가까운 문서 벡터 최대 3개
)

# 5. 자연어 query 입력 -> 유사도 높은 Document 반환 받기
retriever_documents = retriever.invoke('벡터 데이터베이스란?')

retriever_documents

[Document(id='63b99c5d-0f35-47c5-963c-c0a03b9c9ba9', metadata={'author': 'jin', 'source': 'https://vectorbase.net', 'topic': 'vector'}, page_content='벡터 데이터베이스는 임베딩된 데이터를 효율적으로 검색할 수 있습니다.'),
 Document(id='16c7a86a-2b21-4b99-b6c6-56597fe18a96', metadata={'author': 'jin', 'source': 'https://vectorbase.net', 'topic': 'vector'}, page_content='벡터 데이터베이스는 임베딩된 데이터를 효율적으로 검색할 수 있습니다.'),
 Document(id='25768d92-9602-4a74-a5a9-66a14b90058a', metadata={'date': '2024-07-01', 'license': 'MIT', 'source': 'https://chromadb.org/intro'}, page_content='ChromaDB는 오픈소스 벡터 데이터베이스입니다.')]

### 기존 Vector Store에 문서 추가하기

`add_documents()`는 새 `Document`를 현재 Pinecone index에 추가한다.

- 입력: 추가할 `list[Document]`이다.
- 변환: 각 `page_content`를 같은 embedding 모델로 변환한다.
- 출력: 저장된 레코드의 ID 목록이다.
- 다음 사용처: 반환 ID로 레코드를 갱신하거나 삭제할 수 있다.

이 호출도 원격 쓰기를 수행한다. 반복 실행 전에는 같은 문서가 이미 저장되어 있는지 확인한다.


In [16]:
new_documents=[
    Document(
        page_content='파인콘은 클라우드에서만 실행 가능하다',
        metadata={'source': 'https://naver.com'}
    )
]

# 연결된 Pinecone index에 문서 추가
# -> 문서 벡터화 수행 후 index에 추가
# -> 결과로 레코드 식별ID가 반환됨
added_document_ids = vector_store.add_documents(documents=new_documents)
print(added_document_ids)

['d8cbfeda-b8ec-4485-9473-7b9ae0987285']


## Wine Magazine 리뷰 인덱싱

이제 [Wine Reviews 데이터셋](https://www.kaggle.com/datasets/christopheiv/winemagdata130k)의 실제 리뷰를 검색 문서로 사용한다.

`CSV 행 → Document → embedding vector → winemag-review-data index`

### `CSVLoader`가 만드는 값

- `page_content`: CSV 열 이름과 값이 줄 단위로 들어간다.
- `metadata['source']`: 원본 CSV 경로가 들어간다.
- `metadata['row']`: 원본 행 번호가 들어간다.

전체 데이터는 약 13만 행이다. 다운로드 시간, OpenAI embedding 비용, Pinecone 저장 시간, 중복 적재 가능성을 확인한 뒤 실행한다.


### 수업용 CSV 내려받기

`gdown`은 공개 Google Drive 파일을 내려받는다.

- 입력: 수업용 Wine Magazine CSV의 파일 ID이다.
- 출력: 현재 작업 폴더의 `winemag-data-130k-v2.csv`이다.
- 다음 사용처: `CSVLoader`가 이 파일의 각 행을 읽는다.

다운로드가 끝나면 출력에 표시된 파일명과 저장 경로를 확인한다.


In [17]:
# !gdown 11OTMjkDgCSixGKSCXON6iijYysUg2_cq

Downloading...
From: https://drive.google.com/uc?id=11OTMjkDgCSixGKSCXON6iijYysUg2_cq
To: C:\SKN_AI\09_llm\06_rag\winemag-data-130k-v2.csv

  0%|          | 0.00/52.9M [00:00<?, ?B/s]
  1%|          | 524k/52.9M [00:00<00:52, 990kB/s]
  2%|▏         | 1.05M/52.9M [00:00<00:36, 1.42MB/s]
  3%|▎         | 1.57M/52.9M [00:00<00:25, 1.99MB/s]
  5%|▍         | 2.62M/52.9M [00:01<00:14, 3.53MB/s]
  6%|▌         | 3.15M/52.9M [00:01<00:13, 3.80MB/s]
  9%|▉         | 4.72M/52.9M [00:01<00:07, 6.05MB/s]
 11%|█         | 5.77M/52.9M [00:01<00:08, 5.70MB/s]
 13%|█▎        | 6.82M/52.9M [00:01<00:07, 6.35MB/s]
 15%|█▍        | 7.86M/52.9M [00:01<00:06, 6.58MB/s]
 17%|█▋        | 8.91M/52.9M [00:01<00:06, 6.69MB/s]
 19%|█▉        | 9.96M/52.9M [00:02<00:06, 7.08MB/s]
 23%|██▎       | 12.1M/52.9M [00:02<00:04, 8.68MB/s]
 25%|██▍       | 13.1M/52.9M [00:02<00:05, 7.63MB/s]
 30%|██▉       | 15.7M/52.9M [00:02<00:04, 9.14MB/s]
 34%|███▎      | 17.8M/52.9M [00:02<00:03, 10.2MB/s]
 37%|███▋      | 19.4M/

### CSV 행을 `Document` 목록으로 변환하기

`CSVLoader`는 CSV의 한 행을 `Document` 하나로 변환한다.

- `file_path`: 앞에서 내려받은 CSV 경로이다.
- `encoding='utf-8'`: 파일의 문자를 UTF-8 규칙으로 읽는다.
- `load()` 출력: CSV 행 순서를 유지한 `list[Document]`이다.

각 `page_content`에는 열 이름과 값이 줄 단위로 들어간다. 이 단계는 로컬 파일만 읽으며 아직 embedding이나 Pinecone 쓰기를 수행하지 않는다.


In [18]:
from langchain_community.document_loaders import CSVLoader

# 1. csvLoader로 읽을 파일경로와 문자 인코딩 설정
loader = CSVLoader(
    file_path='./winemag-data-130k-v2.csv',
    encoding='utf-8'
)

# 2. CSV 데이터 읽어오기(csv 1행 ->  Document 1개)
docs = loader.load()

# 3. 읽어온 Document 수 확인
print(len(docs)) # 129971




129971


### 변환된 문서 구조 확인하기

전체 문서를 저장하기 전에 앞의 두 `Document`를 확인한다.

- 자료형: `Document`인지 확인한다.
- `page_content`: `description`, `title`, `variety` 같은 리뷰 열이 있는지 확인한다.
- `metadata`: 원본 CSV 경로와 행 번호가 있는지 확인한다.

이 구조가 맞아야 검색 결과에서 리뷰 내용과 원본 행을 함께 추적할 수 있다.


In [19]:
for index, document in enumerate(docs[:2]):
    print(f'{index}: {type(document)}')
    print(document)

0: <class 'langchain_core.documents.base.Document'>
page_content=': 0
country: Italy
description: Aromas include tropical fruit, broom, brimstone and dried herb. The palate isn't overly expressive, offering unripened apple, citrus and dried sage alongside brisk acidity.
designation: Vulkà Bianco
points: 87
price: 
province: Sicily & Sardinia
region_1: Etna
region_2: 
taster_name: Kerin O’Keefe
taster_twitter_handle: @kerinokeefe
title: Nicosia 2013 Vulkà Bianco  (Etna)
variety: White Blend
winery: Nicosia' metadata={'source': './winemag-data-130k-v2.csv', 'row': 0}
1: <class 'langchain_core.documents.base.Document'>
page_content=': 1
country: Portugal
description: This is ripe and fruity, a wine that is smooth while still structured. Firm tannins are filled out with juicy red berry fruits and freshened with acidity. It's  already drinkable, although it will certainly be better from 2016.
designation: Avidagos
points: 87
price: 15.0
province: Douro
region_1: 
region_2: 
taster_name: Rog

### 실제 리뷰를 배치로 적재하기

**Batch**는 전체 문서를 작은 묶음으로 나누어 처리하는 방법이다. 여기서는 `docs`를 100개씩 나누어 `winemag-review-data` index의 기본 namespace에 저장한다.

### 처리 순서

1. 와인 index와 embedding 모델을 연결한다.
2. `range()`와 slice로 `docs`를 100개씩 나눈다.
3. 각 batch를 `add_documents()`에 전달한다.
4. 문서 본문을 embedding하고 Pinecone에 upsert한다.

### 배치를 사용하는 이유

- 한 요청이 사용하는 메모리와 전송량을 제한한다.
- 오류가 발생한 batch 위치를 출력으로 확인할 수 있다.
- 마지막 batch는 100개보다 작아도 그대로 처리할 수 있다.

이 셀은 장시간 실행과 외부 API 비용을 발생시킨다. 재실행하면 이미 저장된 리뷰가 중복될 수 있다.


In [22]:
batch_size = 100

# 1. 임베딩 객체 준비
embeddings = OpenAIEmbeddings(
    model=os.environ['OPENAI_EMBEDDING_MODEL']
)

# 2. 'winemag-review-data' 인덱스와 임베딩 객체를 연결
wine_vector_store = PineconeVectorStore.from_documents(
    documents=[], # 아직 실제 문서를 전달 x
    embedding=embeddings,
    index_name=WINE_INDEX_NAME
)

# 3. 배치 크기만큼으로 docs를 나눠서 추가
for start in range(0, len(docs), batch_size):
    batch = docs[start:start + batch_size]
    print(f'index: {start} batch: {len(batch)}')

    # 4. index에 document 추가
    wine_vector_store.add_documents(documents=batch)



index: 0 batch: 100
index: 100 batch: 100
index: 200 batch: 100
index: 300 batch: 100
index: 400 batch: 100
index: 500 batch: 100
index: 600 batch: 100
index: 700 batch: 100
index: 800 batch: 100
index: 900 batch: 100
index: 1000 batch: 100
index: 1100 batch: 100
index: 1200 batch: 100
index: 1300 batch: 100
index: 1400 batch: 100
index: 1500 batch: 100
index: 1600 batch: 100
index: 1700 batch: 100
index: 1800 batch: 100
index: 1900 batch: 100
index: 2000 batch: 100
index: 2100 batch: 100
index: 2200 batch: 100
index: 2300 batch: 100
index: 2400 batch: 100
index: 2500 batch: 100
index: 2600 batch: 100
index: 2700 batch: 100
index: 2800 batch: 100
index: 2900 batch: 100
index: 3000 batch: 100
index: 3100 batch: 100
index: 3200 batch: 100
index: 3300 batch: 100
index: 3400 batch: 100
index: 3500 batch: 100
index: 3600 batch: 100
index: 3700 batch: 100
index: 3800 batch: 100
index: 3900 batch: 100
index: 4000 batch: 100
index: 4100 batch: 100
index: 4200 batch: 100
index: 4300 batch: 100


KeyboardInterrupt: 

## 직접 적용해 보기

작은 리뷰 범위를 별도의 연습용 namespace에 적재하고 검색하는 코드를 완성한다. 기존 기본 namespace를 반복 적재하지 않도록 새로운 namespace 이름을 먼저 정한다.

1. `docs`에서 앞부분의 제한된 범위를 선택한다.
2. `add_documents()`에 선택한 문서와 연습용 namespace를 전달한다.
3. 같은 index·embedding·namespace를 사용하는 Retriever를 만든다.
4. 음식 풍미 query를 입력하고 반환된 `list[Document]`의 `page_content`와 `metadata`를 확인한다.

완성 조건은 선택한 범위의 실제 리뷰가 검색되고, 검색 Document의 metadata로 원본 CSV 행을 추적할 수 있는 것이다. namespace 값과 검색 개수 `k`는 직접 결정한다.
